In [ ]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [ ]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "Rainbow File"))


In [ ]:
from src.demos.ekg.pydantic_rag import PydanticRag

KV_STORE = "file"


vector_store = PydanticRag.get_vector_store()

rag = PydanticRag(
    model_definition=test_schema,
    vector_store=vector_store,
    llm_id=None,
    kv_store_id=KV_STORE,
)


In [ ]:
import os
from pathlib import Path

doc_id = "03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json"

file1 = Path(os.getenv("ONEDRIVE", "")) / "prj/atos-kg/rainbow-json/" / doc_id
assert file1.exists()
doc_text = file1.read_text()

rainbow_report = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", rainbow_report)

assert rainbow_report


In [ ]:
from src.utils.pydantic.kv_store import save_object_to_kvstore

save_object_to_kvstore(doc_id, rainbow_report)

In [ ]:
debug(rainbow_report)

In [ ]:
chunks = rag.chunck(rainbow_report)
debug(chunks)


In [ ]:
descriptions = []
for field_name, field_info in rag._top_class.model_fields.items():
    description = getattr(field_info, "description", "")
    if description:
        descriptions.append(f"'{field_name}': {description}")


"; ".join(descriptions)

In [21]:
from typing import List, Optional

from langchain_core.documents import Document
from langchain_core.tools import BaseTool
from langchain_core.tools.base import ArgsSchema
from langchain_core.vectorstores import VectorStore
from pydantic import BaseModel, Field

from src.utils.config_mngr import global_config
from src.utils.pydantic.kv_store import save_object_to_kvstore


class VectorSearchInput(BaseModel):
    """Input schema for the vector search tool."""

    query: str = Field(..., description="The search query to find semantically similar documents")
    fields: Optional[List[str]] = Field(
        None, description="Optional list of field names to limit the search to specific fields in the documents"
    )


class VectorSearchTool(BaseTool):
    """Tool for searching the vector store for semantic matches."""

    name: str = "vector_search"
    description:str = "desc"
    args_schema: Optional[ArgsSchema] = VectorSearchInput

    _vector_store: VectorStore = PydanticRag.get_vector_store()

    def _run(self, query: str, fields: Optional[List[str]] = None) -> List[Document]:
        """Execute search against the vector store."""
        filter_dict = {}
        if fields:
            # Create filter for specific fields
            filter_dict = {"field_name": {"$in": fields}}

        return self._vector_store.similarity_search(query, k=4, filter=filter_dict)
    
tool = VectorSearchTool()

TypeError: cannot pickle 'module' object

In [ ]:
rag.create_vector_search_tool()

In [ ]:
# 2. Index the document
rag.store_chunks(chunks)
print("Document stored.")


In [ ]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

In [ ]:
# 3. Query the vector store
hits = rag.query_vectorstore("revenue", k=2, filter={"field_name": {"$eq": "financials"}})
print("Vector hits:", hits)